# BinX AI & ML Internship
## Week 4 — Day 1: Train / Validation / Test Splits

**Topics covered:**
- The problem with tuning against a single test set
- The three-way split: train, validation, test
- Creating a three-way split in code
- Why one validation set can still mislead (motivating cross-validation)

---

##  Where I'm Starting From

In Week 3, every experiment followed the same pattern:

```
dataset → train/test split → model.fit(X_train) → model.score(X_test)
```

That felt complete. But today I realized it has a hidden problem.

If I trained three models and picked the one with the best test score, I used the test set to make a decision. The moment I do that, the test score stops being an honest estimate — it becomes a score I optimized for, without realizing it.

Today is about fixing that.

---

## The Three Way Split

Each set has exactly one job and should never do another set's job.

| Set | Purpose | When It's Used |
|-----|---------|----------------|
| **Training** | Model learns its parameters | During `.fit()` |
| **Validation** | Tune — model type, hyperparameters, features | During development, as many times as needed |
| **Test** | Final, one-time honest performance estimate | Once, at the very end — never touched during tuning |

**Train to learn. Validate to choose. Test to report.**

---

### 1. Training Set — Learning

The training set is the portion of the dataset the model learns from directly.

When I call `model.fit(X_train, y_train)`, the model sees the features and
the correct answers together, then adjusts its internal values to minimize
its prediction error. Those internal values are called **parameters** —
for example, the coefficients in Logistic Regression, or the splitting
rules in a Decision Tree.

This is the only set the model's parameters are learned from.
The validation and test sets never touch `.fit()`.

**Why a high training score isn't always good news**

A model can score F1 = 1.00 on training data and still be a poor model.
That happens when it memorized the training examples — including the noise —
instead of learning the real pattern. It performs perfectly on data it has
seen and fails on anything new. That's overfitting, and training performance
alone can't detect it.

> Training = Learning

---

### 2.Validation Set — Decision Making

The validation set exists because development involves decisions, not just training.

After fitting a model, I often need to choose between options:

- Which model type performs better?
- Which `max_depth` should I use?
- Should I keep or drop a feature?

I can't use the test set for these comparisons — that would compromise it.
And I can't use the training set either, because the model has already seen it,
so any score on training data is biased.

The validation set is a slice of data the model has never trained on,
reserved specifically for making development decisions. I can check it as
many times as I need — once per experiment, dozens of times — because it
isn't the final evaluation.

**The connection to overfitting**

The validation set is also the tool that catches overfitting during development.
If a model scores very high on training data but noticeably lower on validation,
that gap is the signal that the model memorized rather than generalized.

**One limitation**

If I run hundreds of experiments against the same validation set, my decisions
can start adapting to that specific slice of data. This is called
*validation overfitting*, and it's the motivation for cross-validation tomorrow.

> Validation = Decision Making

---

### 3.Test Set — Final Evaluation

The test set is held out until the very end, after every decision is final —
model type chosen, hyperparameters set, features selected.

It gets used exactly once.

**Why only once?**

Every time I look at the test score and adjust something based on it,
I'm leaking information from the test set into my decisions. The score
starts reflecting how well I optimized for those specific rows, not how
well the model generalizes to truly new data. That's *information leakage*,
and it makes the final score dishonest.

An "honest estimate" means: the test set had zero influence on any decision
I made. The score I get is the closest thing I have to the model's
real-world performance.

> Test = Final Evaluation

---

### Why Can't I Just Use Train + Test?

Here's the scenario that breaks the two set approach:

```text
Model A → Test F1 = 0.81
Model B → Test F1 = 0.84  ← I choose this
Model C → Test F1 = 0.79
```

This looks like a fair comparison. But by choosing Model B because of its
test score, I just used the test set to make a decision. It's no longer
independent of my development process.

The same problem appears with hyperparameter tuning:

```text
max_depth = 3  → Test F1 = 0.74
max_depth = 5  → Test F1 = 0.79  ← I choose this
max_depth = 10 → Test F1 = 0.76
```

Every check is a leak. By the end, the "final" score reflects how well
I optimized for those specific test rows — not how the model performs on
data it's never seen in any form.

The solution is to never let the test set answer development questions.
That's the validation set's job.

---

### Why Validation Can Still Mislead

The validation set protects the test set — but it has its own limitation.

A small number of comparisons on one validation set is usually fine.
But if I run hundreds of experiments against the same split:

```text
Experiment 1   → Val F1 = 0.64
Experiment 2   → Val F1 = 0.67
...
Experiment 100 → Val F1 = 0.73  ← I choose this
```

By that point, my choices have started fitting the quirks of that particular
154-row validation slice. The selected configuration might look better than
it actually is — because it happened to suit that slice, not because it
genuinely generalizes better.

This is called *validation overfitting*.

The fix is to evaluate across multiple different train/validation splits
instead of relying on one. That's cross-validation, and it's tomorrow's topic.

---

## How the Three Way Split Works

```text
Full Dataset (768 rows)
│
├── 20% → Test set       (154 rows)  ← locked immediately
│
└── 80% → Temporary      (614 rows)
          │
          ├── 75% → Training set    (460 rows)
          └── 25% → Validation set  (154 rows)
```

The math: 80% × 75% = 60% of the original → Training
          80% × 25% = 20% of the original → Validation

Scikit-learn's `train_test_split` creates two groups at a time,
so a three-way split always takes two calls — the first carves out
the test set, the second divides what remains.

---

## Putting It Into Practice

Now that I understand the role of each set and why we need to keep the
test set untouched, I'll put the three-way split into practice using the
Pima Diabetes dataset from Week 3.

I'll first divide the data into **training, validation, and test sets**,
then run a small hyperparameter tuning experiment using the **validation
set only**. Finally, I'll evaluate the selected model on the **test set**
to see how the complete process works in practice.

### 1.Imports Libraries

In [80]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

SEED = 42

### 2.Loading the Data

I'll use the same Pima Diabetes dataset from Week 3.

It contains **768 samples** and represents a binary classification task:

- `Outcome = 1` → diabetic
- `Outcome = 0` → not diabetic

The CSV has no header row, so I assign the column names manually.

In [81]:
col_names = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

df = pd.read_csv(
    "../../Week3/Day 3/Data/pima-indians-diabetes.csv",
    header=None,
    names=col_names
)

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["Outcome"].value_counts())
print(f"\nPositive class: {df['Outcome'].mean():.1%}")

Shape: (768, 9)

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Positive class: 34.9%


> Note :I'm using the raw Pima dataset here (768 rows, no cleaning applied)
because the focus of this notebook is the split workflow, not data quality.

### 3.Creating the Three Way Split

A three-way split requires **two calls** to `train_test_split`.

1. First, I set aside **20% as the final test set**.
2. Then, I split the remaining **80% into 75% training and 25% validation**.

This gives an overall split of approximately:

**60% Training / 20% Validation / 20% Test**

I also use `stratify` in both splits to approximately preserve the
original class distribution across all three sets.

In [82]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Step 1 — lock away 20% as the final test set immediately
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# Step 2 — split the remaining 80% into train (75%) and validation (25%)
# 80% × 75% = 60% of the original → Training
# 80% × 25% = 20% of the original → Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp
)

print(
    f"Training set:   {X_train.shape[0]} rows  "
    f"({X_train.shape[0] / len(X):.0%})"
)

print(
    f"Validation set: {X_val.shape[0]} rows  "
    f"({X_val.shape[0] / len(X):.0%})"
)

print(
    f"Test set:       {X_test.shape[0]} rows  "
    f"({X_test.shape[0] / len(X):.0%})  ← locked"
)

print(
    f"\nTotal: "
    f"{X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}"
)

Training set:   460 rows  (60%)
Validation set: 154 rows  (20%)
Test set:       154 rows  (20%)  ← locked

Total: 768


### 4.Verifying Class Balance

Because this is a classification problem, I want to make sure that the
three splits have a similar proportion of diabetic cases.

The original dataset contains about **34.9% positive cases**.

Using `stratify` should keep this proportion approximately consistent
across the training, validation, and test sets.

A large difference would make one split less representative of the
original class distribution.

In [83]:
print("Diabetic proportion in each split:")

print(f"  Full dataset:   {y.mean():.3f}")
print(f"  Training set:   {y_train.mean():.3f}")
print(f"  Validation set: {y_val.mean():.3f}")
print(f"  Test set:       {y_test.mean():.3f}")

Diabetic proportion in each split:
  Full dataset:   0.349
  Training set:   0.348
  Validation set: 0.351
  Test set:       0.351


All four proportions are nearly identical, which shows that stratify
preserved the class distribution well.

### 5.Tuning on Validation Only

I'll try three `max_depth` values for a Random Forest and compare
them on the validation set only. The test set stays locked.

The interesting thing to watch: `max_depth=10` and `None` will get
near-perfect training F1 — but their validation F1 drops. That gap
is overfitting, and the validation set is what catches it.

In [84]:
results = {}

for depth in [3, 10, None]:
    model = RandomForestClassifier(
        max_depth=depth,
        n_estimators=100,
        random_state=SEED
    )

    model.fit(X_train, y_train)

    train_f1 = f1_score(
        y_train,
        model.predict(X_train)
    )

    val_f1 = f1_score(
        y_val,
        model.predict(X_val)
    )

    results[depth] = {
        "train_f1": train_f1,
        "val_f1": val_f1
    }

    print(
        f"max_depth={str(depth):>4}  →  "
        f"Train F1: {train_f1:.3f}  |  "
        f"Val F1: {val_f1:.3f}"
    )

max_depth=   3  →  Train F1: 0.689  |  Val F1: 0.674
max_depth=  10  →  Train F1: 0.991  |  Val F1: 0.626
max_depth=None  →  Train F1: 1.000  |  Val F1: 0.646


### 6.Choosing the Best Hyperparameter

I choose the `max_depth` with the **highest validation F1**.

The training score is not used to make this decision because the model
has already learned from the training data.

In the results above, `max_depth=3` achieves the highest validation F1.
The deeper models achieve much higher training F1, but their validation
F1 is lower, suggesting that they are fitting the training data too
closely.

This is a practical example of why validation performance matters more
than training performance when making development decisions.

In [85]:
# Pick the depth with the highest validation F1
best_depth = max(
    results,
    key=lambda d: results[d]["val_f1"]
)

print(f"Best max_depth (chosen by validation): {best_depth}")
print(
    f"Validation F1 at this setting:         "
    f"{results[best_depth]['val_f1']:.3f}"
)

Best max_depth (chosen by validation): 3
Validation F1 at this setting:         0.674


### 7.Final Evaluation on the Test Set 

All development decisions are now finished.

I selected `max_depth` using the validation set, so I can now evaluate
the final model on the test set.

This is the **first and only time** I use the test set in this experiment.

The test score is not used to change the model or choose another
hyperparameter. It is only used to estimate how the selected model
performs on unseen data.

In [86]:
final_model = RandomForestClassifier(
    max_depth=best_depth,
    n_estimators=100,
    random_state=SEED
)

final_model.fit(X_train, y_train)

test_predictions = final_model.predict(X_test)

test_f1 = f1_score(y_test, test_predictions)
test_acc = accuracy_score(y_test, test_predictions)

print(
    f"Validation F1:  {results[best_depth]['val_f1']:.3f} "
    f"(used for selection)"
)

print(
    f"Test F1:        {test_f1:.3f} "
    f"(final honest estimate)"
)

print(f"Test Accuracy:  {test_acc:.3f}")

Validation F1:  0.674 (used for selection)
Test F1:        0.562 (final honest estimate)
Test Accuracy:  0.727


---

## What I Learned From the Experiment

The validation F1 was **0.674**, while the final test F1 was **0.562**.
This difference is possible because the validation and test sets contain
different samples.

The important point is not that the two scores must be identical.
The important point is that the test set had **no influence on any
development decision**.

The experiment followed the intended workflow:

- **Training →** the model learned its parameters here.
- **Validation →** I compared hyperparameter choices here.
- **Test →** I evaluated the final selected model here.

The results also showed why training performance alone is not enough.
The deeper Random Forest models achieved very high training F1, but their
validation F1 was lower. This is a practical signal of overfitting.

### Remaining Limitation

The validation set is still only one slice of the data — in this case,
154 samples. It could be somewhat lucky or unlucky.

Cross-validation will address this limitation by evaluating the model
across multiple rotating validation folds, giving a more reliable estimate
during development.
